# pdbKit in a notebook
The same functions the `pdbkit` command uses, called from Python. Structures are Biopython `Structure` objects, so pdbKit mixes with any Biopython code.

In [1]:
import warnings; warnings.filterwarnings('ignore', message='You may be importing Biopython')
import pdbkit as pk
pk.__version__

'1.0.0'

In [2]:
s = pk.load('../../tests/data/4HHB.cif')     # a PDB ID such as '4HHB' is fetched and cached
pk.summary(s)

{'id': '4HHB',
 'models': 1,
 'chains': ['A', 'B', 'C', 'D'],
 'polymer_residues': 574,
 'ligand_residues': 6,
 'water_molecules': 221,
 'atoms': 4779,
 'hetatm_atoms': 395,
 'has_altlocs': False,
 'has_insertion_codes': False,
 'has_sequence_gaps': False,
 'ligands': {'HEM': 4, 'PO4': 2}}

## Split chains: C-alpha trace with the haem groups

In [3]:
parts = pk.split_chains(s, atoms='ca', het=['HEM'], water=False)
{c: len(list(p.get_atoms())) for c, p in parts.items()}

{'A': 184, 'B': 189, 'C': 184, 'D': 189}

In [4]:
paths = pk.write_parts(parts, 'nb_out', '4HHB', 'pdb', 'ca_het-HEM')
paths

['nb_out/4HHB_A_ca_het-HEM.pdb',
 'nb_out/4HHB_B_ca_het-HEM.pdb',
 'nb_out/4HHB_C_ca_het-HEM.pdb',
 'nb_out/4HHB_D_ca_het-HEM.pdb']

## Centre of mass of chain A, moved to the origin (whole tetramer kept)

In [5]:
centred, com = pk.center(s, chains='A', apply_to='all')
print('COM before:', com.round(3), ' after:', pk.center_of_mass(centred, chains='A').round(6))
pk.save(centred, 'nb_out/4HHB_chainA_com0.cif')

COM before: [23.724 18.531 29.704]  after: [ 0.  0. -0.]


'nb_out/4HHB_chainA_com0.cif'

## Tables as pandas DataFrames

In [6]:
pk.to_dataframe(pk.contacts(s, 'HEM', 3.5)).head()

,target,residue,min_distance
0,A:HEM:142,A:TYR:42,3.268
1,A:HEM:142,A:HIS:45,2.976
2,A:HEM:142,A:HIS:58,3.366
3,A:HEM:142,A:LYS:61,3.486
4,A:HEM:142,A:LEU:86,3.498


In [7]:
pk.to_dataframe(pk.sasa(s)).sort_values('sasa', ascending=False).head()

,residue,sasa,relative
547,D:LYS:120,204.04,0.865
260,B:LYS:120,181.58,0.769
493,D:LYS:66,178.13,0.755
206,B:LYS:66,165.31,0.700
429,D:HIS:2,163.67,0.731


## FASTA for every chain

In [8]:
print(pk.fasta(s))

>4HHB_A mol:protein length:141 source:seqres Hemoglobin subunit alpha
VLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGK
KVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPA
VHASLDKFLASVSTVLTSKYR
>4HHB_B mol:protein length:146 source:seqres Hemoglobin subunit beta
VHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKV
KAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGK
EFTPPVQAAYQKVVAGVANALAHKYH
>4HHB_C mol:protein length:141 source:seqres Hemoglobin subunit alpha
VLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGK
KVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPA
VHASLDKFLASVSTVLTSKYR
>4HHB_D mol:protein length:146 source:seqres Hemoglobin subunit beta
VHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKV
KAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGK
EFTPPVQAAYQKVVAGVANALAHKYH



## Domain / family / secondary-structure table (offline: no UniProt/KEGG lookups)

In [9]:
pk.to_dataframe(pk.domains.annotate(s, skip_lookups=True))[['Chain_ID','Entity_Description','UniProt_Accession','Heteroatoms_In_Chain','Heteroatom_Bonding_Partners']]

,Chain_ID,Entity_Description,UniProt_Accession,Heteroatoms_In_Chain,Heteroatom_Bonding_Partners
0,A,Hemoglobin subunit alpha,P69905,HEM142,"HEM142 -> HIS87(chain A, metalc)"
1,B,Hemoglobin subunit beta,P68871,PO4147; HEM148,PO4147 -> (no _struct_conn record found); HEM1...
2,C,Hemoglobin subunit alpha,P69905,HEM142,"HEM142 -> HIS87(chain C, metalc)"
3,D,Hemoglobin subunit beta,P68871,PO4147; HEM148,PO4147 -> (no _struct_conn record found); HEM1...
